In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv.ipython import load_dotenv

In [77]:
load_dotenv(override=True)

True

In [3]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [9]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpfull assistant"
)

In [10]:
resp=agent.invoke(input={"message":[
    {"role":"user","content":"Je m'appelle Abdo"}
]})

In [11]:
print(resp['messages'][-1].content)

Of course! How can I assist you today?


In [12]:
resp=agent.invoke(input={"message":[
    {"role":"user","content":"Comment je m'appelle?"}
]})

In [22]:
from langchain_core.messages import HumanMessage

In [43]:
basic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
advanced_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [44]:
def select_model(env: str):
    if env == "prod":
        model = advanced_llm
    else :
        model = basic_llm
    return model

In [45]:
def agent2(input, context=None):
    context = context or {}
    env = context.get("env", "test")

    model = select_model(env)

    messages = input["messages"]

    return model.invoke(messages)

In [46]:
resp = agent2(
    input={"messages": [HumanMessage(content="C'est quoi un agent AI")]},
    context={"env": "prod"}
)

In [47]:
from IPython.display import Markdown

In [49]:
display(Markdown(resp.content))

Un agent AI (agent d'intelligence artificielle) est un système informatique conçu pour percevoir son environnement, prendre des décisions et agir de manière autonome ou semi-autonome afin d'atteindre des objectifs spécifiques. Les agents AI peuvent être simples ou complexes, et ils sont utilisés dans une variété de domaines, tels que :

1. **Assistants virtuels** : Comme Siri, Alexa ou Google Assistant, qui aident les utilisateurs à effectuer des tâches quotidiennes en répondant à des questions, en envoyant des messages, ou en contrôlant des appareils connectés.

2. **Agents conversationnels** : Aussi connus sous le nom de chatbots, ils interagissent avec les utilisateurs via des interfaces de chat pour fournir des informations ou des services.

3. **Systèmes de recommandation** : Utilisés par des plateformes comme Netflix ou Amazon pour suggérer des films, des séries ou des produits en fonction des préférences de l'utilisateur.

4. **Agents autonomes** : Comme les voitures autonomes, qui perçoivent leur environnement et prennent des décisions de conduite sans intervention humaine.

5. **Robots industriels** : Utilisés dans les chaînes de production pour effectuer des tâches répétitives avec précision.

6. **Agents de trading** : Utilisés dans le secteur financier pour analyser les marchés et exécuter des transactions de manière autonome.

Les agents AI reposent souvent sur des techniques d'apprentissage automatique, de traitement du langage naturel, de vision par ordinateur, et d'autres domaines de l'intelligence artificielle pour fonctionner efficacement.

In [50]:
from langgraph.checkpoint.memory import InMemorySaver

In [53]:
memory = InMemorySaver()
agent3 = create_agent(
    model = "openai:gpt-4o",
    system_prompt="You are a helpfull asistant",
    checkpointer = memory
)

In [54]:
config = {"configurable":{"thread_id":1}}
resp = agent3.invoke(
    input={"messages":[HumanMessage("Je m'appelle Abdo")]},
    config=config
    )

In [55]:
print(resp["messages"][-1].content)

Bonjour Abdo ! En quoi puis-je vous aider aujourd'hui ?


In [58]:
resp = agent3.invoke(
    input={"messages":[HumanMessage("Comment je m'appelle?")]},
    config=config
    )

In [59]:
print(resp["messages"][-1].content)

Vous m'avez dit précédemment que vous vous appelez Abdo.


In [60]:
from langchain.tools import tool

In [61]:
@tool
def get_weather(city : str):
    """
    Get The weather of the given city
    """
    print(f"Weather Tool invoked")
    return {
        "city":city,
        "temperature":21,
        "humidity":77,
        "pressure":110
    }

In [62]:
@tool
def get_employee_info(employee_name : str):
    """
    Get infos about the given employee
    """
    print(f"Info employee tool invoked")
    return {
        "name":employee_name,
        "salary":21000,
        "seniority":7
    }

In [64]:
agent4 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather,get_employee_info],
    checkpointer=memory,
    system_prompt="answer the user question using only provided tools"
    )

In [65]:
config = {"configurable":{"thread_id":1}}
resp=agent4.invoke(
    input={'messages':[HumanMessage("La météo à Rabat")]},
    config=config
)

Weather Tool invoked


In [66]:
print(resp['messages'][-1].content)

La météo à Rabat indique une température de 21°C avec une humidité de 77% et une pression atmosphérique de 110 hPa.


In [67]:
config = {"configurable":{"thread_id":1}}
resp=agent4.invoke(
    input={'messages':[HumanMessage("Quel est le salaire de Hassan?")]},
    config=config
)

Info employee tool invoked


In [68]:
print(resp['messages'][-1].content)

Hassan a un salaire de 21 000 et compte 7 ans d'ancienneté dans l'entreprise.


In [69]:
load_dotenv(override=True)

True

In [70]:
from ddgs import DDGS

In [71]:
@tool
def web_searchDDGS(query: str, num_results:int=5) -> str:
    """
    Search the web usin DuckDuckGo
    Args:
    query : Search query string
    num_results : Number of results to return (Default=5)
    Returns:
    Formatted search results with titles, descptions and Urls
    """
    try:
        print(f'Search tool is called for {query}')
        ddgs_search = DDGS()
        results=ddgs_search.text(query=query, max_results=num_results, backend="google")
        if not results:
            return f"No results found for {query} "
        formatted_results = [f"Search for {query} : \n"]
        for i, result in enumerate(results,1):
            title = result.get("title","No Title")
            body = result.get("body","No description available")
            href = result.get("href","")
            formatted_results.append( f"{i}. **{title}**. {body}. {href}")
        return formatted_results
    except Exception as e:
        print(str(e))

In [72]:
agent5 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather,get_employee_info, web_searchDDGS],
    checkpointer=memory,
    system_prompt="answer the user question using only provided tools",
    debug=True
    )

In [73]:
rep=agent5.invoke(input={"messages":[HumanMessage("Search for python tutorials")]},
              config=config
              )

[values] {'messages': [HumanMessage(content="Je m'appelle Abdo", additional_kwargs={}, response_metadata={}, id='9b8b37ce-ed92-4dd3-92f9-ba51d1b0f2f2'), AIMessage(content="Bonjour Abdo ! En quoi puis-je vous aider aujourd'hui ?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 23, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_9dd8023e2f', 'id': 'chatcmpl-DYW91TANdhy3fRD4X1svzuzwRqqGD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc48d-415d-7fb0-b690-4c05b185856c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 23, 'output_tokens': 13, 'total_tokens': 36, 'input_token_details': {'audio': 0, '

In [74]:
print(display(Markdown(resp['messages'][-1].content)))

Hassan a un salaire de 21 000 et compte 7 ans d'ancienneté dans l'entreprise.

None


In [75]:
from langchain_tavily import TavilySearch

In [79]:
tavily = TavilySearch(max_results=10, search_depth="advanced")

In [80]:
@tool
def search_webTavily(query:str):
    """
    Search for general current web results
    """
    print(f"search_web invoked with {query}")
    results = tavily.invoke({"query":query})
    return results

In [82]:
agent6 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather,get_employee_info, search_webTavily],
    checkpointer=memory,
    system_prompt="answer the user question using only provided tools"
    )

In [85]:
resp=agent6.invoke(input={"messages":[HumanMessage("Atualités sur le maroc")]},
              config=config
              )

search_web invoked with actualités sur le Maroc


In [86]:
print(display(Markdown(resp['messages'][-1].content)))

Voici quelques actualités récentes concernant le Maroc :

1. **Football Africain :** Le Maroc a été déclaré vainqueur de la CAN 2026 par la Confédération Africaine de Football (CAF) après que le Sénégal ait été déchu de son titre à cause du comportement de ses joueurs.

2. **Inondations à Safi :** Des crues violentes à Safi ont causé la mort d'au moins 37 personnes. Ces intempéries sont considérées comme liées au réchauffement climatique.

3. **Vie de Famille au Maroc :** Une enquête révèle que les familles marocaines se marient moins, divorcent plus et vivent dans des logements plus petits. Ces changements soulignent une transformation sociale profonde.

4. **Essor du Tourisme :** Le Maroc a accueilli 8,9 millions de touristes au premier semestre 2025, et le gouvernement se prépare pour les élections législatives de 2026.

Pour plus de détails, vous pouvez consulter les articles disponibles sur [Franceinfo](https://www.franceinfo.fr/monde/afrique/maroc/) et [Jeune Afrique](https://www.jeuneafrique.com/pays/maroc/).

None


In [87]:
from langchain_experimental.tools import PythonREPLTool

In [88]:
repl_tool = PythonREPLTool(sanitize_input=False)

In [89]:
agent7 = create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="""Générer et exécuter le code python
    en plançant le code python générer et le résultat  d'execution de ce code dans le fichier doc.txt
    """,
    debug=True
)

In [90]:
resp=agent7.invoke(
    input={"message":[HumanMessage("Je veux trier les deux listes [6,8,2,1] et [5,1,9,4] et puis additionner les deux listes")]}
)

Python REPL can execute arbitrary code. Use with caution.


[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 239, 'prompt_tokens': 110, 'total_tokens': 349, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d59908c1a9', 'id': 'chatcmpl-DYWG95wtVWvkfhZXnHZfXYud9GLtz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc494-0395-7413-8559-27442434e6e8-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'code = """\n# This is a simple Python code for adding two numbers\ndef add_numbers(a, b):\n    return a + b\n\nresult = add_numbers(5, 3)\nprint(f\'The result of adding 5 and 3 is: {result}\')\n"""\n\nwith open(\'doc.txt\', \'w\') as file:\n    file.write(\'Pyth

In [91]:
print(resp['messages'][-1].content)

The Python code and execution result have been successfully placed into the file `doc.txt`. Here is what was written:

### Python Code:
```python
# This is a simple Python code for adding two numbers
def add_numbers(a, b):
    return a + b

result = add_numbers(5, 3)
print(f'The result of adding 5 and 3 is: {result}')
```

### Execution Result:
```
The result of adding 5 and 3 is: 8
```


In [92]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

In [93]:
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        print(f"Exception {e}")
    return ToolMessage(
        content=f"Tool error: Please check your input and try again. ({str(e)})",
        tool_call_id=request.tool_call["id"]
    )

In [94]:
agent8 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather],
    middleware=[handle_tool_errors], debug=True
)

In [98]:
resp=agent8.invoke(
    input={"messages":[HumanMessage("Quelle est la météo en erreur?")]}
)

[values] {'messages': [HumanMessage(content='Quelle est la météo en erreur?', additional_kwargs={}, response_metadata={}, id='e1012ed6-bdb8-48ca-ab4e-ee753320f355')]}
[updates] {'model': {'messages': [AIMessage(content='Pouvez-vous préciser de quelle ville ou région vous souhaitez connaître la météo ?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 52, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_9dd8023e2f', 'id': 'chatcmpl-DYWJa0EAdERjhoAaaMWzjf5rfbm2n', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc497-465c-7af2-a8db-6776bc809048-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'ou

In [99]:
print(resp['messages'][-1].content)

Pouvez-vous préciser de quelle ville ou région vous souhaitez connaître la météo ?


In [100]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

In [101]:
class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str: 
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."
    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."
    
    return base_prompt

In [102]:
agent9 = create_agent(
    model="openai:gpt-4o",
    tools=[],
    middleware=[user_role_prompt],
    context_schema=Context
)

In [103]:
resp=agent9.invoke(
    input={"messages":[HumanMessage("Merci de m'expliquer la Machine Learning")]},
    context={"user_role": "beginner"}
)

In [104]:
print(resp['messages'][-1].content)

Bien sûr ! Le machine learning, ou apprentissage automatique en français, est une branche de l'intelligence artificielle. Elle permet aux ordinateurs d'apprendre à partir de données plutôt que d'être programmés de manière explicite pour accomplir une tâche précise.

Voici les bases :

1. **Données** : Tout commence par des données que l'on collecte. Par exemple, si on veut qu’un programme reconnaisse des images de chats, on commence par lui donner de nombreuses images de chats (et d'autres animaux pour qu'il puisse faire la différence).

2. **Modèle** : Le modèle est comme une formule complexe ou un algorithme que l'ordinateur utilise pour essayer de faire des prévisions ou des décisions. Au début, le modèle ne sait rien, mais il va s'améliorer au fur et à mesure qu'il verra plus de données.

3. **Entraînement** : C'est le processus par lequel le modèle, avec l'aide des données, ajuste ses paramètres pour améliorer ses prévisions ou décisions. Cela signifie apprendre à mieux reconnaîtr